<a href="https://colab.research.google.com/github/Nav-del/Machine-Learning-Course/blob/main/AI_Agent_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Below code shows all the models the API Key Supports


In [ ]:
import google.generativeai as genai
genai.configure(api_key="#API_KEY") #hiding key
available_models = genai.list_models()
list(available_models)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


[Model(name='models/gemini-2.5-flash',
       base_model_id='',
       version='001',
       display_name='Gemini 2.5 Flash',
       description=('Stable version of Gemini 2.5 Flash, our mid-size multimodal model that '
                    'supports up to 1 million tokens, released in June of 2025.'),
       input_token_limit=1048576,
       output_token_limit=65536,
       supported_generation_methods=['generateContent',
                                     'countTokens',
                                     'createCachedContent',
                                     'batchGenerateContent'],
       temperature=1.0,
       max_temperature=2.0,
       top_p=0.95,
       top_k=64),
 Model(name='models/gemini-2.5-pro',
       base_model_id='',
       version='2.5',
       display_name='Gemini 2.5 Pro',
       description='Stable release (June 17th, 2025) of Gemini 2.5 Pro',
       input_token_limit=1048576,
       output_token_limit=65536,
       supported_generation_methods=['generateCon

In [ ]:
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI #to name the specifications of the llm
from langchain_core.messages import HumanMessage, AIMessage
import re

In [ ]:
pip install langchain

In [ ]:
pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
gemini_api_key = "#API_KEY" #hiding the key
genai.configure(api_key=gemini_api_key)

Just to understand, how to get Gemini model outputs here

In [ ]:
import google.generativeai as genai
genai.configure(api_key="#API_KEY") #hiding the key
#select model
model = genai.GenerativeModel("gemini-2.0-flash")

#Now make a value prompt and add a question there
prompt= "Who is prime minister of India"

#Now generate response
response = model.generate_content(prompt)

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 55.493276734s.

In [ ]:
#now if you want only the text output
response.text

In [ ]:
eval("2+3*5") #Use of Eval() is to evaluate anythin inside the brackets

17

In [ ]:
# initialize LLM
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.0-flash", #selecting the model
    google_api_key=gemini_api_key,  #adding the api key
    temperature=0, #strict mode
    max_output_tokens=512   #capping tokens
)

# Temperature ranges from (0 - 1)


# **temp-0 (do not get creative give me standard replies)**
# is earth flat ? -> no as per studies earth is a globe
# are bees important for humans? -> yes bees are important for human

# **temp-1 (get creative give me standard replies)**
# is earth flat ? -> well, good quyestion! allow me to repon tj
# are bees important for humans? -> yes bees are important for human


Now we need to tell the llm which Actions to perform

In [ ]:
def calculate(expression):
  return eval(expression)

def average_pill_weight(name):
  pill_weights = {
      "Aspirin":"500mg",
      "Parcetamol":"650mg",
      "Ibuprofen" : "400mg"
  }
  if name in pill_weights:
      return pill_weights[name]
  return "Unknown pill"

known_actions = {
    "calculate": calculate,
    "average_pill_weight": average_pill_weight
}
#The actions the agent can perform are calculate and average_pill_weight

In [ ]:
class Agent:
  def __init__(self):
    self.messages =[]

  #using method overriding
  def  __call__(self,message):
    self.messages.append(HumanMessage(content=message)) #to segregate Human and AI messages
    response = self.execute() #execute function will be made later in the code
    self.messages.append(AIMessage(content=response)) #To get the response recieved from the AI
    return response

  def execute(self):
    response = llm.invoke(self.messages)
    return response.content

In [ ]:
action_re = re.compile(r'Action:\s*(\w+):\s*(.*?)(?:\n|$)', re.MULTILINE)
#symbols are regex anchors

In [ ]:
def run_query(prompt, max_turns=10): #max_turns lets the agent work on the input 10 times, if it still fails, program stops
  agent = Agent() #we already have a class Agent given there
  next_prompt = prompt

  #to let the user know number of turns
  for turn in range(max_turns):
    print(f"Turn{turn+1}:")
    response = agent(next_prompt)
    print(f"AI: \n{response}\n")

    actions = action_re.findall(response)
    if actions: # Check if actions list is not empty
      action, action_input =  actions[0]
      action_input = action_input.strip()

      if action not in known_actions:
        raise ValueError(f"Unknown action: {action}")

      observation = known_actions[action](action_input)
      print(f"Observation: {observation}\n")
      next_prompt = f"Observation: {observation}"
    else: # No action found
      if "Answer:" in response:
          print(f"Final Answer provided!")
          break # Now this break is correctly inside the for loop's logical flow.
      else:
          print(f"No action found and no final answer. Stopping.") # Corrected "Stoping" to "Stopping"
          break # This break is also correctly inside the for loop's logical flow.

instruction_and_question = """
You are a helpful assistant following a ReAct reasoning pattern.

For each step:
1. Use "Thought:" to think step by step about the problem
2. If you need more information, use 'Action: <action_name>:<input>' on a new line
3. After each action. I will provide an 'Observation:' with the result
4. Continue thinking and acting until you can provide a final 'Answer:'

Available actions:
- calculate: perform a python calculation (e.g., calculate: 5 + 3)
- average_pill_weight : Get the average_pill_weight of a specific pill (e.g., average_pill_wei

Important : Only perform ONE action at a time, then wait for the observation.
Question : I have two pills, Aspirin and Paracetamol,
what is their combined weight?
"""

In [ ]:
run_query(instruction_and_question)

Turn1:


ChatGoogleGenerativeAIError: Error calling model 'models/gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 29.554123s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 

*****

# Initializing Agent Steps
*   Check RAG diagram



In [ ]:
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.agents import initialize_agent, Tool, AgentType

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ModuleNotFoundError: No module named 'langchain_google_genai'

In [ ]:
#Now copy Gemini API key
gemini_api_key = "#API_KEY" #hiding the key
genai.configure(api_key=gemini_api_key)

#Now initialize the llm variable
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.0-flash", #selecting the model
    google_api_key=gemini_api_key,  #adding the api key
    temperature=0, #strict mode
    max_output_tokens=512   #capping tokens
)

In [ ]:
#Now we make, the content present inside the variable document
documents = ['HARISH LOVES TO EAT APPLES AND ORANGES',
             "HARIS IS AN EXTROVERT, LOVES TO TRAVEL AND HAS A DOG NAMES CASPER",
             "HARIS IS A PERSON OF GOOD CHARACTER AND HIGH INTEGRITY",
             "HARISH DOES NOT LIKE WHEN PEOPLE LIE"]
#This could be in any format, Dictionary, list, database anything

#To print the content in the variable document
doc = [Document(page_content=doc) for doc in documents]
doc #to print

NameError: name 'Document' is not defined

*Output*

[Document(metadata={}, page_content='HARISH LOVES TO EAT APPLES AND ORANGES'),

 Document(metadata={}, page_content='HARISH IS AN EXTROVERT, LOVES TO TRAVEL AND HAS A DOG NAMES CASPER'),

 Document(metadata={}, page_content='HARISH IS A PERSON OF GOOD CHARACTER AND HIGH INTEGRITY'),

 Document(metadata={}, page_content='HARISH DOES NOT LIKE WHEN PEOPLE LIE')]


**Why do we need to do this?**

When a sentence is given, we not only save it as it is but try to make a table which has Info like -> Text | Embedding | Metadata

The embedding: embedded text

Metadata: Has info about the text (like who sent it, through what, reason for this sentence)

This is hard manually so a library:

***from langchain.docstore.document import Document***

is called which does this.

In the above code, we didnt pass any metadata in the doc variable.

So in the output **metadata={}** is empty




In [ ]:
# embedding model - text -> emb
#This is the model we use to embed the text from a tool named HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Now we need to convert it to embedding and then store it in Vector DB

In [ ]:
vectorStore = FAISS.from_documents(docs,embeddings) #internal DB
#FAISS is a free vector DB where we store our content
#This DB has 2 columns, docs and then embedding

Now we match Embedding with the Questions embedding (notebook concept)

In [ ]:
def search_documents(query):    #Taking user query and putting it into the function
    results = vectorstore.similarity_search(query, k=2) #top two similar Embeddings to the Question Embeddings are given in the variable "results"
    combined = "\n".join([f"- {res.page_content}" for res in results])
    return f"Relevant Documents:\n{combined}"

**combined = "\n".join([f"- {res.page_content}" for res in results])**

We initialize a variable *combined* :

**{res.page_content}** : The result of page_content (present in the output of the cell in doc code)

So what happens here is:

1. We fisrt check the "similarity_search" which gives the 2 most similar strings
2. Those 2 strings are present in the "results" variable
3. Those strings are then joined in the "combined" variable
4. It joins the "page_content" of the previos doc code that we did
5. Then returns as a single Result

In [ ]:
#To find the similarity between the Question and the Answer
search_tool = Tool(
    name = "VectorStoreLookup",
    description = "useful for when you need to find information about a specific context from documents",
    func = search_documents
)

#To find the similarity between the Question and the Answer
search_tool = Tool( -> We had imported tool in the import cell earlier
    
    
    #We need the agent to look into the vector DB
    name = "VectorStoreLookup" -> This is the name of the tool, nothing else


    #We are making the agent search the DB, so now we add the description
    description = "useful for when you need to find information about a specific context from documents",
    
    
    func = search_documents
)

In [ ]:
agent = initialize_agent(
    tools = [search_tool], #We can add multiple tools here
    llm = llm, #Already initialized previously
    agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION, #Here we tell Langchain to make a ReAct Agent (skipping the step done in cell 11)
)

In [ ]:
instructions = (
    "You are a research assistant for AI"
    "When asked a question, think step-by-step"
    "If you need more info, use the VectorStoreLookup tool by writing your thought process, then call the tool"
    "After getting the info, summarize it and provide a final concise answer to the user"
)

In [ ]:
query = f"{instructions}\n What is reinforcement learning?"
#Query is whatever instructions we have + the users question
response = agent.run(query)

OUTPUT

**reponse**

'Reinforcement Learning (RL) is a machine learning method where an agent learns optimal behaviors by interacting with an environment through trial and error.'


Here, the agent gets the info from the database which stores the "doc" which was added earlier. (Internal Database)

In [ ]:
#Now if we ask a Q not present in the database
query = f"{instructions}\n When did India win the last worldcup?"
#Query is whatever instructions we have + the users question
response = agent.run(query)

OUTPUT

**reponse**

'India has won the Cricket World Cup twice, in 1983 and 2011.'


Here this data was not found in the internal database, so it goes to the internet and gives the answer (fallback mechanism)

This is because of Absence of Guard Rail againt Gemini, which will answer anything you ask according to the models capability.

# How do you solve this now?
In instructions add another line like:

"If the user question has nothing to do with the tool or relevance, do not respond and say it is irrelevant."


In [ ]:
instructions = (
    "You are a research assistant for AI"
    "When asked a question, think step-by-step"
    "If you need more info, use the VectorStoreLookup tool by writing your thought process, then call the tool"
    "After getting the info, summarize it and provide a final concise answer to the user"
    "If the user question has nothing to do with the tool or relevance, do not respond and say it is irrelevant."
)

In [ ]:
#Now if we ask a Q not present in the database
query = f"{instructions}\n When did India win the last worldcup?"
response = agent.run(query)

OUTPUT

response

"It's irrelevant"

*****

# RAG
with fallback

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted

ModuleNotFoundError: No module named 'langchain_community'

In [4]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


HuggingFaceEmbeddings & FAISS: Used to generate vector representations of your local documents and search through them based on semantic similarity.

TavilySearchResults: A specialized search tool that allows your AI agent to query the live internet for up-to-date facts if your local documents don't have the answer.

genai: Google's official SDK used to initialize and interact with Gemini LLM models for text generation.

ResourceExhausted: An error handling exception used to catch API rate-limiting issues (like hitting free-tier token per minute limits) so your application can handle them gracefully instead of crashing.

Save API keys in the key icon +Add new Secret


In [1]:
transformer_text = """ Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures.

Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [21] and conditional computation, while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains.
"""

Since here the text is too big, for the LLM to understand we perform **Chunking**

chunking : Making small chunks of the paragraph and then feeding the model.

But by doing this we do not know wheich para comes after which.

So we do

Overlap-Chunking : Take chunks and the next chunk has part of the previous chunk for cintinuity

In [5]:
def chunk_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        if len(chunk)>200:
            chunks.append(chunk)
    return chunks or [text]

def chunk_text(text, chunk_size=500):  # Defines a function to split long text into smaller pieces

    chunks = []  # Initializes an empty list to store the extracted text blocks

    for i in range(0, len(text), chunk_size):  # Loops through the text index in steps equal to the chunk size

        chunk = text[i:i+chunk_size]  # Slices out a specific segment of text based on the current index step

        if len(chunk)>200:  # Checks if the extracted text segment is longer than 200 characters

            chunks.append(chunk)  # Adds the text segment to our list if it passes the length check

    return chunks or [text]  # Returns the gathered chunks, or a list containing the original full text if empty


In [6]:
chunk_text(transformer_text)
#running the function

[' Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures.\n\nRecurrent models typically factor computation along the symbol positions of the input and output sequence',
 's. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [21] and cond

In [ ]:
def summarize_with_retry(text, max_retries = 3):
    if not text.strip():
        return "Empty Content"
    prompt = f"Summarize concisely:\n{text}"
    for attempt in range(max_retries):
        try:
            response = model.generate_content(prompt)
            return response.text.strip() if response.text else "No summary generated"
        except ResourceExhausted:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Rate limit hit, waiting {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                return "Error: API quota exceeded"
    return "Error: Failed after retries"

This function sends text to an AI model to generate a concise summary while safely handling API rate limits. If a quota error occurs, it automatically pauses and retries the request up to three times using an exponential backoff delay. If all attempts fail or the input text is completely empty, it returns a clean error string instead of crashing your application.

In [ ]:
def create_vector_store(texts):
    summaries = [summarize_with_retry(text) for text in texts]

    valid_summaries = [s for s in summaries if not s.startswith("Error")]

    if not valid_summaries:
        raise ValueError("No valid summaries generated for vectorstore")

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    documents = [Document(page_content=summary, metadata={"id":str(uuid.uuid4())}) for summary in valid_summaries]

    return FAISS.from_documents(documents, embeddings)


This function takes a list of texts, generates summaries for them, and automatically filters out any failed API results. It converts the valid summaries into structured LangChain documents assigned with unique UUID tracking tags. Finally, it uses a HuggingFace model to calculate vector embeddings for these documents and saves them into a searchable FAISS database.

In [ ]:
def answer_question(query, vector_store):
    print(f"Question : {query}")

    if vector_store:
        try:
            docs_with_scores = vector_store.similarity_search(query, k=3)
            relevant_docs = [doc for doc, score in docs_with_scores if score < 0.8]
            if relevant_docs:
                context = "\n".join(doc.page_content for doc in relevant_docs)
                prompt = f"Answer based on this context : \n{context}\n\nQuestion:{query}"
                response = model.generate_content(prompt)
                if response.text:
                    return {"answer":response.text.strip(), "source":"internal document"}
        except Exception as e:
            print(f"RAG failed : {str(e)}")
    try:
        search_results = search_tool.invoke({"query":query})
        return {"Answer": str(search_results), "source":"internet search"}
    except Exception as e:
        print(f"Error : {str(e)}")


This function attempts to answer a user's question by first searching a local FAISS database and filtering for highly relevant document matches below a specific distance threshold. If the local search yields no quality matches or fails, it automatically falls back to an online search tool to find the answer on the internet. Finally, it returns the generated answer bundled along with a dictionary key indicating whether the information came from an internal document or a web search.

In [ ]:
def main():
    vector_store = None
    try:
        chunks = chunk_text(transformer_text)
        vector_store = create_vector_store(chunks)
        print(f"Vector store created successfully ")
    except Exception as e:
        print(f"Vector store creation failed")
        print(f"Will use use internet search")

questions = [
    "who is the prime minister of trinidad and tobago?",
    "To facilitate these residual connections what is the model dimenstion mentiond in the paper"
]

for question in questions:
    result = answer_question(question, vector_store)
    print(f"Answer: {result['answer']}")
    print(f"Source: {result['source']}")


This is the entry point script that orchestrates the entire RAG pipeline by chunking a reference text (the Transformer paper) and initializing the FAISS vector store database. It loops through a set of test questions—one general knowledge query and one paper-specific query—and passes them to the routing engine to be answered. Finally, it prints out the model's generated response alongside the verification source (local document vs. internet search).

OUTPUT

Vector store created successfully
Question : who is the prime minister of trinidad and tobago?
docs_with_scores : [Document(id='b6f44e42-689a-4705-bbce-d4dcde708436', metadata={'id': '64affef2-97ca-4941-97d8-4ca89'})]

RAG failed : too many values to unpack (expected 2)

Answer: [{'title': 'List of prime ministers of Trinidad and Tobago', 'url': 'https://wikipedia.org'}]

Source: internet search


Question : The decoder is also composed of a stack of how many identical layers.
docs_with_scores : [Document(id='286c6775-02c5-453f-823b-fcae3de293c6', metadata={'id': 'e4943004-2b8b-4eb8-ad9d-e5d60'})]

RAG failed : too many values to unpack (expected 2)

Answer: [{'title': 'The decoder stack in the Transformer model | by Sandaruwan Herath', 'url': 'https://medium.com'}]

Source: internet search


**MCP : **

Easy-to-Understand Analogy:

The USB-C Cable ->

Imagine a world where every single device required a totally unique plug:

Your mouse used an Apple-specific wire.

Your keyboard used an external Dell-specific wire.

Your printer required a custom HP adapter block.

If you bought a new computer, none of your old devices would plug in without writing new software or buying custom breakout boxes.

**USB-C solved this by creating one standard shape and protocol that everyone agreed to use. **

Now, you plug a USB-C cable into any laptop, and it instantly works.MCP is the USB-C port for Artificial Intelligence.

............................................

Instead of teaching an LLM how to talk to a custom company database using specialized code, the database uses an MCP Server wrapper, and the AI connects to it seamlessly via its MCP Client.

*****